# GeneticLab - pruebas iniciales de MOEA

Este notebook prueba optimizadores tipo `ask/tell` contra el pipeline existente de `genes`.

La idea es repetir este ciclo:

1. `ask`: el optimizador propone una poblacion.
2. `dispatch`: se generan casos y se corre Elmer/Gmsh con la reserva Slurm actual.
3. `build_objectives`: se convierten resultados a objetivos y constraints.
4. `tell`: el optimizador recibe los resultados y avanza una generacion.

Por defecto todo queda en `GeneticLab/experiments/<RUN_LABEL>`.

## 0. Configuracion

En el cluster, deja `PROJECT = Path('/work/jmorera/Genes')`. Esta carpeta `GeneticLab` debe estar al mismo nivel que `genes`, `Genetic1`, `minicycle` y `testparallel_alloc`.

Empieza con `DRY_RUN = True` para revisar comandos. Cuando la previsualizacion se vea bien, cambia a `False`.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import subprocess
import sys

import pandas as pd

DRY_RUN = True

# Ruta raiz del proyecto en el cluster.
PROJECT = Path('/work/jmorera/Genes').resolve()
LAB_ROOT = PROJECT / 'GeneticLab'

# Cambia ALGORITHM para comparar variantes.
# Opciones: age2, rvea, ctaea, sms, nsga3, random
ALGORITHM = 'age2'
POP_SIZE = 16
SEED = 11
MAX_GENERATIONS = 250  # RVEA lo necesita para adaptar sus vectores de referencia.

# Primera prueba: 5 puntos en dz, 1 individuo por nodo, 20 MPI ranks por individuo.
MAX_NODES_TO_USE = 4
INNER_NPROCS = 20
N_POINTS = 5
START_MM = -3.0
END_MM = 3.0

RUN_LABEL = f'{ALGORITHM}_gtest_' + datetime.now().strftime('%Y%m%d_%H%M%S')
EXPERIMENT_ROOT = LAB_ROOT / 'experiments' / RUN_LABEL
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

STATE_PKL = EXPERIMENT_ROOT / 'optimizer_state.pkl'
ASK_POP_CSV = EXPERIMENT_ROOT / 'ask_population.csv'
ARCHIVE_CSV = EXPERIMENT_ROOT / 'optimizer_archive.csv'
OPT_EVAL_CSV = EXPERIMENT_ROOT / 'results' / '05_optimizer' / 'optimizer_evaluation.csv'

print('PROJECT:', PROJECT)
print('LAB_ROOT:', LAB_ROOT)
print('EXPERIMENT_ROOT:', EXPERIMENT_ROOT)
print('DRY_RUN:', DRY_RUN)

## 1. Helpers

Estas celdas llaman scripts con `python3`. El script de dispatch tambien tiene modo seco: si no le pasamos `--execute`, imprime los comandos externos sin correr Elmer/Gmsh.

In [ ]:
MOEA_CYCLE_PY = LAB_ROOT / '40_optimizacion' / 'moea_cycle.py'
DISPATCH_PY = LAB_ROOT / '20_ejecucion' / 'dispatch_existing_pipeline.py'
BUILD_OBJECTIVES_PY = LAB_ROOT / '30_postproceso' / 'build_objectives.py'
SPACE_JSON = LAB_ROOT / '00_config' / 'design_space_default.json'

def sh(cmd, cwd=None, execute=True):
    if not execute:
        return 'DRY_RUN local command:\n' + cmd
    proc = subprocess.run(['bash', '-lc', cmd], cwd=str(cwd) if cwd else None, capture_output=True, text=True)
    chunks = []
    if proc.stdout:
        chunks.append('STDOUT:\n' + proc.stdout)
    if proc.stderr:
        chunks.append('STDERR:\n' + proc.stderr)
    if proc.returncode != 0:
        chunks.append(f'[returncode={proc.returncode}]')
    return '\n'.join(chunks)

for path in [MOEA_CYCLE_PY, DISPATCH_PY, BUILD_OBJECTIVES_PY, SPACE_JSON]:
    print(path, '->', path.exists())

## 2. Elegir algoritmo

- `age2`: recomendacion inicial general. Bueno para many-objective sin definir preferencias.
- `rvea`: bueno cuando quieres guiar el frente con referencias.
- `ctaea`: interesante si las restricciones empiezan a dominar.
- `sms`: util si te quedas en 2-3 objetivos; hipervolumen.
- `nsga3`: baseline moderno por referencias, no NSGA-II.
- `random`: smoke test sin depender de `pymoo`.

Si `pymoo` no esta instalado, prueba primero `random` o instala `requirements_optional.txt`.

In [ ]:
ask_cmd = f"""
python3 {MOEA_CYCLE_PY} ask \
  --state {STATE_PKL} \
  --out {ASK_POP_CSV} \
  --algorithm {ALGORITHM} \
  --pop-size {POP_SIZE} \
  --seed {SEED} \
  --max-generations {MAX_GENERATIONS} \
  --space {SPACE_JSON}
"""

# Este paso es liviano; normalmente conviene ejecutarlo aunque DRY_RUN sea True.
print(sh(ask_cmd, cwd=LAB_ROOT, execute=True))

population_df = pd.read_csv(ASK_POP_CSV)
display(population_df.head())
print('Rows:', len(population_df))

## 3. Correr el pipeline existente

Esta celda llama a `genes/10_poblacion`, `genes/20_ejecucion` y `genes/30_postproceso` por medio del wrapper de GeneticLab.

Con `DRY_RUN = True`, veras los comandos. Con `DRY_RUN = False`, el wrapper agrega `--execute` y corre realmente.

In [ ]:
execute_arg = '--execute' if not DRY_RUN else ''
dispatch_cmd = f"""
python3 {DISPATCH_PY} \
  --project {PROJECT} \
  --lab-root {LAB_ROOT} \
  --population {ASK_POP_CSV} \
  --run-label {RUN_LABEL} \
  --start-mm {START_MM} \
  --end-mm {END_MM} \
  --n-points {N_POINTS} \
  --max-nodes {MAX_NODES_TO_USE} \
  --nprocs {INNER_NPROCS} \
  {execute_arg}
"""

print(sh(dispatch_cmd, cwd=PROJECT, execute=True))

## 4. Revisar objetivos

El wrapper intenta crear `optimizer_evaluation.csv`. Si tus columnas reales de `population_evaluation.csv` no coinciden con los aliases, edita `GeneticLab/30_postproceso/build_objectives.py` en el diccionario `ALIASES`.

Regla para `pymoo`: todos los objetivos se minimizan y todas las constraints deben ser `<= 0`.

In [ ]:
if OPT_EVAL_CSV.exists():
    opt_eval_df = pd.read_csv(OPT_EVAL_CSV)
    display(opt_eval_df.head())
    print(opt_eval_df[['objective_primary', 'objective_abs_force0', 'objective_nonlin', 'objective_volume_m3', 'constraint_eval_failed', 'constraint_geometry']].describe())
else:
    print('Aun no existe:', OPT_EVAL_CSV)
    print('Corre el dispatch real o construye objetivos manualmente cuando tengas population_evaluation.csv.')

## 5. Tell al optimizador

Ejecuta esta celda solo cuando `optimizer_evaluation.csv` exista y tenga todas las filas de la poblacion pendiente.

In [ ]:
if OPT_EVAL_CSV.exists():
    tell_cmd = f"""
python3 {MOEA_CYCLE_PY} tell \
  --state {STATE_PKL} \
  --evaluation {OPT_EVAL_CSV} \
  --archive-out {ARCHIVE_CSV}
"""
    print(sh(tell_cmd, cwd=LAB_ROOT, execute=True))
else:
    print('No hago tell porque falta optimizer_evaluation.csv')

## 6. Mirar archivo acumulado

Para comparar algoritmos, repite el notebook cambiando `ALGORITHM` y `RUN_LABEL`. En una prueba justa, usa el mismo `POP_SIZE`, mismo presupuesto total y misma malla coarse.

In [ ]:
if ARCHIVE_CSV.exists():
    archive_df = pd.read_csv(ARCHIVE_CSV)
    display(archive_df.tail())
else:
    print('Archivo aun no existe:', ARCHIVE_CSV)

## 7. Sugerencia de campana inicial

Orden recomendado:

1. `random`, `POP_SIZE=4` o `8`: smoke test de formato sin gastar mucho.
2. `age2`, `POP_SIZE=16`: primera prueba evolutiva real.
3. `rvea`, `POP_SIZE=16` o `24`: comparar frente con referencias.
4. `ctaea`: si aparecen muchas geometria invalidas o fallos de solver.
5. `sms`: solo si reduces a 2-3 objetivos.

Antes de la corrida grande, valida los mejores no dominados en malla normal/fina y revisa sensibilidad a dominio de aire.